# lunar-scripts-jp

Patch a NieR Re[in]carnation **JP** APK and master data for use with a private server.

仓库：https://github.com/lovelyzy7/lunar-scripts-jp

**How to use:**

1. Fill in the **Configuration** form below.
2. Run cells top-to-bottom, or use **Runtime → Run all**.
3. Patched files will download to your browser.

**Two modes:**

- **Drive mode** (default): Check `use_gdrive`, enter paths relative to `/content/drive/` (e.g. `MyDrive/nierspjp.apk`).
- **URL mode**: Uncheck `use_gdrive`, paste download URLs into `apk_source` / `masterdata_source`.

Each section is independent — skip what you don't need.


In [ ]:
#@title Configuration { display-mode: "form" }
use_gdrive        = True    #@param {type:"boolean"}

#@markdown ---
#@markdown **File Sources** (Drive 相对路径，或 URL)
apk_source        = "MyDrive/nierspjp-3.7.1.apk"          #@param {type:"string"}
apk_save          = "MyDrive/nierspjp-jp-modded.apk"      #@param {type:"string"}
masterdata_source = ""                                     #@param {type:"string"}

#@markdown ---
#@markdown **Server Settings**
grpc_addr         = "203.0.113.10:8003"      #@param {type:"string"}
http_addr         = "203.0.113.10:8080"      #@param {type:"string"}
auth_host         = "203.0.113.10:3000"      #@param {type:"string"}

#@markdown ---
#@markdown **Options**
patch_masterdata  = True     #@param {type:"boolean"}
scripts_branch    = "main"   #@param {type:"string"}


In [ ]:
import os

REPO = "https://github.com/lovelyzy7/lunar-scripts-jp.git"

if use_gdrive:
    from google.colab import drive
    drive.mount("/content/drive")
    def _p(x):
        return x if x.startswith("/") else "/content/drive/" + x
    apk_source, apk_save = _p(apk_source), _p(apk_save)
    if masterdata_source:
        masterdata_source = _p(masterdata_source)
else:
    os.makedirs("/content/src", exist_ok=True)
    if apk_source.startswith("http"):
        local = "/content/src/" + apk_source.split("/")[-1].split("?")[0]
        !wget -q -O "$local" "$apk_source"
        apk_source = local
    if masterdata_source.startswith("http"):
        local = "/content/src/" + masterdata_source.split("/")[-1].split("?")[0]
        !wget -q -O "$local" "$masterdata_source"
        masterdata_source = local

assert os.path.exists(apk_source), "APK 不存在: " + apk_source
print("APK        :", apk_source)
print("输出        :", apk_save)
print("masterdata :", masterdata_source or "(跳过)")
print("服务器      :", grpc_addr, "|", http_addr, "|", auth_host)


In [ ]:
!git clone -q -b $scripts_branch $REPO
%cd lunar-scripts-jp
!pip -q install -r requirements.txt
!python android/build_apk.py tools
!java -version
print("\n[ready]")


In [ ]:
!python android/build_apk.py unpack "$apk_source" work/jp-final


In [ ]:
!python android/patch_apk_jp.py work/jp-final \
    --grpc-addr "$grpc_addr" \
    --http-addr "$http_addr" \
    --auth-host "$auth_host"


In [ ]:
if patch_masterdata and masterdata_source:
    !python masterdata/patch_masterdata.py --input "$masterdata_source" \
        --output "$(dirname "$masterdata_source")/master_data_patched.bin"
else:
    print("[skip] master data patch")


In [ ]:
!python android/build_apk.py build work/jp-final "$apk_save"
import os
print("\n[ok]", apk_save, os.path.getsize(apk_save) // 1048576, "MB")
# 浏览器下载（大文件建议直接用上面的 Drive 路径）：
# from google.colab import files; files.download(apk_save)


### 备注

- Colab 会话是临时的：输入/输出建议都放 Google Drive。
- 不需要 frida（设备侧诊断工具，非必需）。
- APK 工具（apktool + uber-apk-signer）由 `android/build_apk.py tools` 自动下载到 `work/tools/`。
- 主数据补丁输出 `master_data_patched.bin`（与原始 bin 同目录），按需替换服务器使用。
